# Cronache dell'Immortalità ⏳
**Data Storytelling a cura di Salvatore Rudisi**

---

## 🎯 L'Obiettivo Narrativo
Viviamo nell'era dell'iper-consumismo digitale, un'epoca definita dalla "cultura dello scarto" e da una soglia dell'attenzione sempre più frammentata. Siamo abituati a contenuti progettati per essere consumati in un weekend e immediatamente sostituiti dal prossimo trend algoritmico. Eppure, nel panorama degli anime, assistiamo a un fenomeno controintuitivo: opere che sfidano le leggi del mercato, restando sulla cresta dell'onda per decenni.

Mentre migliaia di titoli affogano nel dimenticatoio dopo appena 12 episodi, alcuni franchise sembrano aver trovato la formula per l'immortalità commerciale. Ma è davvero solo merito di budget infiniti e "inerzia" produttiva, o c'è un'alchimia profonda che lega l'evoluzione della trama alla fedeltà generazionale del pubblico?

In questa analisi indagheremo la longevità come resilienza, cercando di capire cosa permetta a un anime di trasformarsi da semplice prodotto d'intrattenimento a pilastro culturale inamovibile.

**Esploreremo:**
1. Storia della produzione
2. I Segreti della Sopravvivenza
3. Il "Club dei Decennali": Qualità o Inerzia?
4. Il cimitero degli spettatori
5. I Titani del Presente

In [249]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ast
import sys
import os
from pathlib import Path
import plotly.express as px

# Add project root to sys.path
current_dir = Path(os.getcwd())
# Assuming we are in solution/notebooks or root. 
# Depending on where this notebook runs, we need to find 'solution' package.
if 'solution' not in os.listdir(current_dir):
    # Try moving up
    if 'solution' in os.listdir(current_dir.parent):
        sys.path.append(str(current_dir.parent))
    elif 'solution' in os.listdir(current_dir.parent.parent):
        sys.path.append(str(current_dir.parent.parent))

try:
    from solution.src.data import load_processed
    print("✅ Library solution.src.data loaded.")
except ImportError:
    print("⚠️ Could not import solution.src.data. Please ensure you are running from the correct directory.")



✅ Library solution.src.data loaded.


In [250]:
# Configuration
pd.set_option('display.max_columns', None)
px.defaults.template = "plotly_dark"

### 🛠️ Preparazione dei Dati
Per studiare l'evoluzione e la sopravvivenza degli anime, abbiamo bisogno di dati puliti e coerenti.
In questa fase tecnica andiamo a:
1.  **Caricare e Unire** le informazioni anagrafiche (`details`) con le statistiche del pubblico (`stats`).
2.  **Calcolare il Tempo:** Trasformiamo le date grezze in formati utilizzabili per calcolare la durata esatta di ogni show.
3.  **Filtrare il Rumore:** Poiché ci interessano le *serie* e la loro longevità, escludiamo Film, OAV singoli e opere non pertinenti. Vogliamo analizzare solo le storie che si sono sviluppate nel tempo.

In [251]:
# Data Loading
print("Loading data...")
try:
    df_details = load_processed("details")
    df_stats = load_processed("stats")
    print(f"Details loaded: {df_details.shape}")
    print(f"Stats loaded: {df_stats.shape}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Fallback/Debug note
    
# Data Cleaning & Preparation
print("Cleaning data...")

# 1. Parse Dates
df_details['start_date'] = pd.to_datetime(df_details['start_date'], errors='coerce')
df_details['end_date'] = pd.to_datetime(df_details['end_date'], errors='coerce') # Importante: errors='coerce' trasforma errori in NaT

# Extract Start Year
df_details['start_year'] = df_details['start_date'].dt.year

# Calculate Duration (Time) in Days
df_details['duration_days'] = (df_details['end_date'] - df_details['start_date']).dt.days

# 2. Parse Genres and Studios
def safe_eval(x):
    if isinstance(x, list):
        return x
    try:
        if pd.isna(x): return []
        return ast.literal_eval(str(x))
    except:
        return []
df_details['genres_list'] = [safe_eval(x) for x in df_details['genres']]
df_details['studios_list'] = [safe_eval(x) for x in df_details['studios']]

# 3. Handle Episodes
df_details['episodes'] = pd.to_numeric(df_details['episodes'], errors='coerce')

# 4. Merge
cols_to_merge = ['anime_id', 'watching', 'completed', 'dropped', 'plan_to_watch', 'total']
df = df_details.merge(df_stats[cols_to_merge], on='anime_id', how='left')

# 5. Rimuoviamo i tipi non pertinenti
if 'type' in df.columns:
    types_to_exclude = ['Movie', 'Music', 'Special', 'CM', 'PV']
    df = df[~df['type'].isin(types_to_exclude)]
    
# --- 6. PULIZIA DATE ---
print(f"Righe prima della pulizia date: {len(df)}")

# Logica: Rimuoviamo chi ha status "finito" (cioè NON Currently Airing) ma data di fine mancante.
# In altre parole, teniamo SOLO chi:
# A) È ancora in corso ('Currently Airing')
# B) Ha una data di fine valida (.notna())
df = df[
    (df['status'] == 'Currently Airing') | (df['end_date'].notna())
]
print(f"Righe dopo la pulizia date: {len(df)}")

print(f"Data cleaning complete.")
df.head(2)

Loading data...
Details loaded: (28955, 29)
Stats loaded: (28939, 27)
Cleaning data...
Righe prima della pulizia date: 17528
Righe dopo la pulizia date: 10706
Data cleaning complete.


,anime_id,title,title_japanese,url,image_url,type,status,score,scored_by,start_date,end_date,synopsis,rank,popularity,members,favorites,genres,studios,themes,demographics,source,rating,episodes,season,year,producers,explicit_genres,licensors,streaming,start_year,duration_days,genres_list,studios_list,watching,completed,dropped,plan_to_watch,total
7,299,.hack//Liminality,.hack//LIMINALITY,https://myanimelist.net/anime/299/hack__Limina...,https://cdn.myanimelist.net/images/anime/7/230...,OVA,Finished Airing,6.58,17130,2002-06-20 00:00:00+00:00,2003-04-10 00:00:00+00:00,"While playing the newly released MMORPG ""The W...",6740,4178,34986,35,['Mystery'],['Bee Train'],['Video Game'],[],Original,PG-13 - Teens 13 or older,4,,2002,"['Bandai Visual', 'Bandai', 'CyberConnect2']",[],['Bandai Entertainment'],[],2002.0,294.0,[Mystery],[Bee Train],858.0,24478.0,998.0,7504.0,34987.0
8,9332,.hack//Quantum,.hack//Quantum,https://myanimelist.net/anime/9332/hack__Quantum,https://cdn.myanimelist.net/images/anime/1894/...,OVA,Finished Airing,7.11,20412,2011-01-28 00:00:00+00:00,2011-04-07 00:00:00+00:00,"Tobias, Mary, and Sakuya challenge the impregn...",3960,3808,42217,72,"['Action', 'Adventure', 'Fantasy']",['Kinema Citrus'],['Video Game'],[],Original,PG-13 - Teens 13 or older,3,,2011,"['Bandai Visual', 'flying DOG', 'Showgate', 'B...",[],['Funimation'],['Crunchyroll'],2011.0,69.0,"[Action, Adventure, Fantasy]",[Kinema Citrus],1156.0,28997.0,683.0,10752.0,42223.0


## Capitolo 1: Storia della Produzione 📈
Prima di misurare la *durata* delle vite, contiamo le *nascite*. L'animazione giapponese è passata da prodezza artigianale a industria di massa. 
Osservando il numero di nuove serie lanciate ogni anno, possiamo tracciare le "Golden Age" e l'eventuale saturazione del mercato odierno. Siamo di fronte a una bolla produttiva?



In [252]:
# Grafico 1: Evoluzione Temporale del Numero di Produzioni
# Contiamo semplicemente quante opere sono iniziate in ogni anno
production_counts = df['start_year'].value_counts().sort_index().reset_index()
production_counts.columns = ['start_year', 'count']

# Filtriamo per rimuovere anni con dati sporchi o troppo futuri (es. range 1960-2025)
production_counts = production_counts[
    (production_counts['start_year'] >= 1960) & 
    (production_counts['start_year'] <= 2025)
]

fig1 = px.bar(production_counts, x='start_year', y='count',
              title='Grafico 1: L\'Esplosione Produttiva - Anime Prodotti per Anno',
              labels={'start_year': 'Anno di Inizio', 'count': 'Numero di Anime'},
              # 'Plasma' o 'Turbo' garantiscono che anche i valori bassi siano luminosi e visibili su sfondo scuro
              color='count', color_continuous_scale='Plasma',
              template='plotly_dark')

fig1.update_layout(xaxis_title="Anno", yaxis_title="Totale Produzioni", bargap=0.1)
fig1.show()

### La Corsa contro il Tempo ⏳
Ma quanto sopravvivono realmente queste opere?
In questo grafico mettiamo alla prova la **longevità**: ogni punto rappresenta una serie. Sull'asse orizzontale vediamo quando è nata, su quello verticale per quanti anni è rimasta (o sta rimanendo) in onda.
Le "scie verdi" che partono dagli anni '70 o '90 e salgono verso l'alto sono i veri immortali: serie che non hanno mai smesso di essere trasmesse.

In [253]:
# Grafico 2: Scatter Plot - Anno di Inizio vs Durata (Anni)
# Risponde alla domanda: "Chi nato nel 1970 è ancora vivo oggi?"

# 1. Preparazione Dati
df_duration = df.copy()

# Gestione Date: Se è in corso (o end_date mancante), usiamo OGGI per calcolare la durata "ad oggi"
df_duration['calc_end_date'] = pd.to_datetime(df_duration['end_date'].fillna(pd.Timestamp.now(tz='UTC')), utc=True)
df_duration['start_date'] = pd.to_datetime(df_duration['start_date'], errors='coerce', utc=True)

# --- NUOVA PARTE: Creiamo una stringa per il tooltip ---
# Se 'end_date' è NaT (Not a Time), scriviamo 'Currently Airing', altrimenti la data formattata YYYY-MM-DD
df_duration['end_date_str'] = df_duration['end_date'].apply(lambda x: x.strftime('%Y-%m-%d') if pd.notnull(x) else 'Currently Airing')

# Calcolo durata in anni
df_duration['years_running'] = (df_duration['calc_end_date'] - df_duration['start_date']).dt.days / 365.25

# FILTRI:
df_duration = df_duration[
    (df_duration['status'] != 'Not yet aired') & 
    (df_duration['years_running'] > 0) & 
    (df_duration['start_year'] >= 1960) & 
    (df_duration['start_year'] <= pd.Timestamp.now().year)
]

# 2. Creazione Grafico
fig2 = px.scatter(df_duration, x='start_year', y='years_running', 
                  color='status', 
                  hover_name='title',
                  # AGGIUNTO: passiamo i dati extra per il tooltip
                  hover_data={'end_date_str': True, 'start_year': True, 'years_running': ':.1f'}, 
                  title='Grafico 2: Sopravvivenza - Durata in Anni per Data di Debutto',
                  labels={'start_year': 'Anno di Debutto', 'years_running': 'Anni', 'status': 'Stato', 'end_date_str': 'Fine'},
                  color_discrete_map={'Currently Airing': '#00CC96', 'Finished Airing': '#EF553B'}, 
                  opacity=0.7,
                  template='plotly_dark')

fig2.update_layout(
    xaxis_title="Anno di Inizio", 
    yaxis_title="Durata Totale (Anni)",
    legend_title_text="Stato Attuale"
)
fig2.show()

In [254]:
# Grafico 3: Box Plot - Distribuzione della Qualità (Score) per Decennio
# Questo grafico è molto meglio della Heatmap per confrontare periodi con numeri di produzione diversi

# 1. Creiamo la colonna Decennio
df['decade'] = (df['start_year'] // 10) * 10

# 2. Filtriamo dati validi (dal 1960 in poi)
df_scored = df.dropna(subset=['score', 'decade'])
df_scored = df_scored[df_scored['decade'] >= 1960]

# Ordiniamo il dataframe in base al decennio così Plotly li mostrerà in sequenza
df_scored = df_scored.sort_values(by='decade') 

# 3. Grafico
fig4 = px.box(df_scored, x='decade', y='score', 
              title='Grafico 3: Il Mito della Nostalgia - Voti per Decennio',
              labels={'decade': 'Decennio', 'score': 'Punteggio Medio'},
              color='decade', # Colora ogni decennio diversamente
              color_discrete_sequence=px.colors.qualitative.Safe, # Colori distinti
              points=False, # Nascondiamo i migliaia di punti singoli per pulizia
              template='plotly_dark')

fig4.update_layout(xaxis_type='category', xaxis_title="Decennio", yaxis_title="Score")
fig4.show()

### 📊 La Chimica dell'Immortalità: Di cosa sono fatti gli anime eterni?
Per capire come cambia il "DNA" di una serie man mano che diventa più longeva, usiamo un grafico di composizione.
Ogni colonna rappresenta una categoria di durata (dalle effimere "Stagionali" alle leggendarie "Eterne").

In [255]:
# Grafico: Composizione Percentuale dei Generi per Fascia di Longevità
# Risolve il problema delle ripetizioni: i generi sono visti come componenti unici delle colonne.

# 1. Preparazione Dati
df_comp = df.explode('genres_list').copy()

# Ricalcolo durata (per sicurezza, se non salvato in df globale)
start = pd.to_datetime(df_comp['start_date'], utc=True, errors='coerce')
end = pd.to_datetime(df_comp['end_date'].fillna(pd.Timestamp.now()), utc=True, errors='coerce')
df_comp['years_running'] = (end - start).dt.days / 365.25

# Fasce Longevità
bins = [0, 1, 2, 5, 10, float('inf')]
labels_dur = ['Stagionale (<1)', 'Breve (1-2)', 'Media (2-5)', 'Lunga (5-10)', 'Eterna (>10)']
df_comp['life_class'] = pd.cut(df_comp['years_running'], bins=bins, labels=labels_dur, right=False)

# Filtriamo solo i Top 10 Generi per evitare un "arcobaleno" illeggibile
top_genres = df_comp['genres_list'].value_counts().head(10).index
df_comp = df_comp[df_comp['genres_list'].isin(top_genres)]
df_comp = df_comp.dropna(subset=['life_class'])

# 2. Calcolo Percentuali
# Contiamo quanti anime di ogni genere ci sono in ogni fascia
comp_counts = df_comp.groupby(['life_class', 'genres_list'], observed=False).size().reset_index(name='count')
# Trasformiamo in percentuale rispetto alla fascia (Totale Colonna = 100%)
# Nota: "observed=False" gestisce i warning sui categorici futuri di pandas
totals = comp_counts.groupby('life_class', observed=False)['count'].transform('sum')
comp_counts['percent'] = comp_counts['count'] / totals

# 3. Grafico Stacked Bar
fig_stack = px.bar(comp_counts, x='life_class', y='percent', color='genres_list',
                   title='Grafico 4: Come cambia il Genere in base alla Durata',
                   labels={'life_class': 'Longevità', 'percent': 'Percentuale', 'genres_list': 'Genere'},
                   # I colori sono assegnati ai generi (univocamente)
                   color_discrete_sequence=px.colors.qualitative.Pastel, 
                   template='plotly_dark')

fig_stack.update_layout(
    yaxis_tickformat='.0%', # Formatta asse Y come %
    xaxis_title="Classe di Longevità",
    yaxis_title="Composizione del Mix",
    legend_title_text="Generi (Top 10)"
)
fig_stack.show()

## Capitolo 2: I Segreti della Sopravvivenza 🧬

La longevità non è solo una questione di quanti episodi riesci a produrre, ma di quanto a lungo riesci a restare rilevante. Alcuni anime bruciano intensamente in 24 episodi ma scompaiono in sei mesi; altri, magari con pause e stagioni distanziate, rimangono in vita per decenni.

In questo capitolo indaghiamo il 'DNA' della sopravvivenza. Esamineremo tre fattori critici:

- **La Fonte**: Essere un Manga offre una garanzia di vita più lunga rispetto a un'opera Originale?
- **Il Genere**: Ci sono generi che strutturalmente si prestano meglio a durare negli anni?
- **Lo Studio**: Esistono case di produzione specializzate nel mantenere in vita i franchise?



### 🔍 Analisi 1: La Fonte (Il Combustibile)

In [256]:
# 2. Creazione Grafico Ibrido (Strip + Box)
# Questo mostra SIA la distribuzione densa (i puntini) SIA le statistiche (il box)
fig_source_time = px.strip(df_clean_duration, 
                           x='duration_years', 
                           y='source',
                           color='source',
                           orientation='h',
                           stripmode='overlay', # Sovrappone i punti
                           title='Grafico 5: Distribuzione della Durata per Fonte',
                           labels={'source': '', 'duration_years': 'Anni di Messa in Onda'},
                           template='plotly_dark',
                           hover_data=['title'])

# Aggiungiamo il Box Plot sopra per dare contesto statistico (mediana/quartili)
fig_source_time.add_trace(
    px.box(df_clean_duration, x='duration_years', y='source', orientation='h', color='source').data[0]
)

# Rimuovi legenda duplicata (il box plot aggiunge tracce extra) e pulisci layout
fig_source_time.update_layout(
    showlegend=False,
    xaxis_title="Anni di Attività",
    yaxis={'categoryorder':'total ascending'},
    height=600,
    # Rendi i box plot semitrasparenti per vedere i punti sotto
    boxgap=0.4
)

# Rendo i punti dello strip plot un po' trasparenti per vedere la densità
fig_source_time.update_traces(marker=dict(opacity=0.5, size=4), selector=dict(type='box', overwrite=False))

fig_source_time.show()

### 🎭 L'Architettura del Genere: Chi invecchia meglio?

In [257]:
# Esplodiamo i generi per analizzarli singolarmente
df_genres_expl = df_clean_duration.explode('genres_list')

# Calcoliamo la durata media per genere
genre_longevity = df_genres_expl.groupby('genres_list')['duration_years'].agg(['mean', 'count']).reset_index()

# Teniamo solo generi con un numero significativo di anime (es. > 50) per evitare distorsioni statistiche
genre_longevity = genre_longevity[genre_longevity['count'] > 50]

# Top 15 generi per durata media
top_long_genres = genre_longevity.sort_values('mean', ascending=True).tail(15)

fig_genre_time = px.bar(top_long_genres, x='mean', y='genres_list', orientation='h',
                        title='Grafico 6: Durata media in anni per Genere',
                        labels={'mean': 'Durata Media (Anni)', 'genres_list': 'Genere'},
                        color='mean',
                        color_continuous_scale='Viridis',
                        template='plotly_dark',
                        text_auto='.1f')

fig_genre_time.update_layout(xaxis_title="Anni Medi di Attività", yaxis_title="")
fig_genre_time.show()

### 🏭 3. I Costruttori di Piramidi: L'Impatto dello Studio

In [258]:
# Esplodiamo gli studi
df_studios_expl = df_clean_duration.explode('studios_list')

# Aggregazione per studio
studio_stats = df_studios_expl.groupby('studios_list').agg(
    avg_durationpy=('duration_years', 'mean'),
    max_duration=('duration_years', 'max'),
    anime_count=('anime_id', 'count')
).reset_index()

# Filtriamo i top studi per numero di produzioni (es. > 20 anime)
top_studios = studio_stats[studio_stats['anime_count'] > 20]

fig_studio_time = px.scatter(top_studios, x='anime_count', y='avg_durationpy',
                             size='max_duration', # La bolla rappresenta l'anime più lungo fatto dallo studio
                             color='avg_durationpy',
                             hover_name='studios_list',
                             title='Grafico 7: Studi per Produzione e Longevità Media',
                             labels={'anime_count': 'Numero Anime Prodotti', 
                                     'avg_durationpy': 'Longevità Media delle Serie (Anni)',
                                     'max_duration': 'Massima Durata Record (Anni)'},
                             template='plotly_dark',
                             color_continuous_scale='Plasma')

fig_studio_time.update_layout(xaxis_type="log", # Scala log per gestire studi giganti come Toei
                              xaxis_title="Numero Anime Prodotti (Log)",
                              yaxis_title="Durata Media dei Progetti (Anni)")
fig_studio_time.show()

## Capitolo 3: Il "Club dei Decennali": Qualità o Inerzia? 🏺

Esiste una soglia psicologica e produttiva fondamentale: i **10 anni**.
Superare un decennio di messa in onda significa aver accompagnato una generazione dall'infanzia all'età adulta. Ma cosa succede alla qualità in questo lasso di tempo?

C'è il sospetto che alcuni anime sopravvivano per pura "inerzia cult": nessuno ha il coraggio di cancellarli, e i fan continuano a guardarli per abitudine ("Sindrome di Stoccolma"), anche se la qualità narrativa è crollata.

In questa sezione isoliamo solo i **Titani (>10 anni di attività)**, sia terminati che ancora in corso, e li mettiamo sotto la lente d'ingrandimento.
1.  **Il confronto crudele:** Un anime decennale ha davvero voti migliori della media o è solo nostalgia?
2.  **La mappa degli Immortali:** Chi sono questi sopravvissuti e dove si posizionano tra capolavori e opere dimenticabili?
3.  **La Top List:** Vediamo in faccia i nomi dei più longevi e giudichiamo il loro punteggio.

In [259]:
# 1. Creiamo il flag "Legendary" (> 10 anni)
df_clean_duration['category'] = df_clean_duration['duration_years'].apply(lambda x: 'Legendary (>10 Anni)' if x >= 10 else 'Standard (<10 Anni)')

# Grafico 1: Box Plot Confronto Diretto
fig_compare = px.box(df_clean_duration, 
                     x='category', 
                     y='score', 
                     color='category',
                     title='Grafico 8: I "Decennali" sono davvero migliori?',
                     labels={'category': 'Categoria', 'score': 'Voto Medio (Score)'},
                     color_discrete_map={'Legendary (>10 Anni)': '#FFA15A', 'Standard (<10 Anni)': '#636EFA'},
                     template='plotly_dark',
                     points=False) # Nascondiamo i punti per vedere meglio le mediane

fig_compare.update_layout(showlegend=False, xaxis_title="")
fig_compare.show()

In [260]:
# Usiamo il Logaritmo per la dimensione
# Sommiamo 1 per evitare log(0) ed enfatizziamo un po' la base
df_titans['log_members'] = np.log10(df_titans['members'] + 1)

fig_titans_scatter = px.scatter(df_titans, 
                                x='duration_years', 
                                y='score', 
                                color='status', 
                                size='log_members', # Usiamo il logaritmo!
                                size_max=25, # Dimensione massima delle bolle controllata
                                hover_name='title',
                                hover_data={'members': True, 'log_members': False}, # Mostriamo il numero vero nel tooltip
                                title='Grafico 9: Chi sopravvive con dignità?',
                                labels={'duration_years': 'Anni di Attività', 'score': 'Voto Medio', 'members': 'Popolarità'},
                                color_discrete_map={
                                    'Finished Airing': '#EF553B',
                                    'Currently Airing': '#00CC96'
                                },
                                opacity=0.8,
                                template='plotly_dark')

fig_titans_scatter.update_layout(
    xaxis_title="Anni di Messa in Onda (Logevità)",
    yaxis_title="Qualità Percepita (Voto)",
    legend_title_text="Stato"
)
fig_titans_scatter.show()

In [261]:
# Prendiamo i Top 20 per durata
top_20_longest = df_titans.sort_values(by='duration_years', ascending=False).head(20)

# Grafico 3: Bar Chart dei Top 20
fig_top20 = px.bar(top_20_longest, 
                   y='title', 
                   x='duration_years', 
                   color='score', # Coloriamo la barra in base al voto!
                   orientation='h', # Orizzontale per leggere i titoli
                   title='Grafico 10: I 20 Anime più longevi della storia',
                   labels={'title': 'Anime', 'duration_years': 'Anni', 'score': 'Voto'},
                   # RdYlGn: Rosso (basso) -> Giallo (medio) -> Verde (alto)
                   color_continuous_scale='RdYlGn', 
                   text='duration_years', # Scriviamo gli anni sulla barra
                   template='plotly_dark')

fig_top20.update_traces(texttemplate='%{text:.1f} anni', textposition='outside')
fig_top20.update_layout(
    yaxis={'categoryorder':'total ascending'}, # Ordina dal meno lungo al più lungo
    xaxis_title="Anni di Trasmissione",
    yaxis_title="",
    height=800 # Un po' più alto per farci stare i nomi
)
fig_top20.show()

## Capitolo 4: Il Cimitero degli Spettatori 💀

Ogni anno in più che una serie rimane in onda è una scommessa sulla pazienza del pubblico.
Il **Drop Rate** (percentuale di utenti che abbandonano la serie) è la misura più onesta della "fatica" narrativa.

Analizziamo il rapporto tra tempo e abbandono su 3 livelli:
1.  **Il Peso del Tempo:** I progetti a lungo termine spaventano il pubblico?
2.  **Le Fasce di Rischio:** Esiste una durata "maledetta" in cui tutti mollano?
3.  **Il Muro dei Titani:** Quali sono i giganti che hanno perso più spettatori lungo la strada?

### 1. Il Paradosso dell'Impegno 📉
Si potrebbe pensare che una serie lunghissima abbia un drop rate più alto per pura stanchezza.
Tuttavia, esiste il fenomeno opposto: il **"Filtro del Fan"**. Chi inizia una serie di 10 anni sa già a cosa va incontro. Spesso sono le serie "medie" (né brevi né infinite) a soffrire di crisi d'identità e perdere pubblico.

Il grafico seguente cerca una correlazione tra Anni di Attività e Percentuale di Abbandono.

In [262]:
# 0. Calcolo Drop Rate (se non presente) e Pulizia
# Drop Rate = (Dropped / Total) * 100
df_clean_duration['drop_rate'] = (df_clean_duration['dropped'] / df_clean_duration['total']) * 100

# Filtriamo dati "sporchi" o con pochissimi utenti totali per evitare percentuali fuorvianti
df_drop = df_clean_duration[df_clean_duration['total'] > 500].copy()

# 1. Grafico Scatter: Durata vs Drop Rate
fig_drop_trend = px.scatter(df_drop, 
                            x='duration_years', 
                            y='drop_rate', 
                            opacity=0.4,
                            trendline="lowess", # Usa una trendline curva per vedere l'andamento non lineare
                            trendline_color_override="cyan",
                            title='Grafico 11: Più dura, più viene abbandonato?',
                            labels={'duration_years': 'Anni di Attività', 'drop_rate': 'Tasso di Abbandono (%)'},
                            hover_data=['title'],
                            template='plotly_dark')

fig_drop_trend.update_layout(
    xaxis_title="Anni di Messa in Onda",
    yaxis_title="Percentuale Utenti che hanno 'Droppato'",
    yaxis_range=[0, 30] # Limitiamo al 30% per vedere meglio (oltre è rumore o serie pessime)
)
fig_drop_trend.show()

### 2. La Zona di Pericolo ⚠️
Non tutte le durate sono uguali.
*   **Stagionali (<1 anno):** Basso rischio, basso impegno.
*   **Medie (1-5 anni):** Il limbo. Abbastanza lunghe da stancare, non abbastanza da diventare icone.
*   **Infinite (>10 anni):** Richiedono fede assoluta.

Il Box Plot qui sotto svela la verità: sorprendentemente, le serie **"Eterne"** hanno spesso tassi di abbandono più bassi (mediana inferiore) rispetto alle serie medie. Perché? Perché chi guarda *One Piece* o *Doraemon* fa parte di una "setta" fedele. Sono le serie di 2-5 anni quelle che rischiano di più di perdere interesse per strada.

In [263]:
# Creiamo le fasce di rischio basate sugli anni
bins = [0, 1, 5, 10, 100]
labels = ['Walk (Stagionali <1)', 'Run (Brevi 1-5)', 'Marathon (Medie 5-10)', 'Ironman (>10)']
df_drop['risk_zone'] = pd.cut(df_drop['duration_years'], bins=bins, labels=labels)

# 2. Violin Plot per Fasce
fig_risk = px.violin(df_drop, 
                     x='risk_zone', 
                     y='drop_rate', 
                     color='risk_zone',
                     box=True, # Mostra boxplot interno
                     points=False,
                     title='Grafico 12: Distribuzione Drop Rate per durata',
                     labels={'risk_zone': 'Tipo di Durata', 'drop_rate': 'Drop Rate (%)'},
                     template='plotly_dark')

fig_risk.update_layout(showlegend=False, xaxis_title="", yaxis_title="Drop Rate (%)")
fig_risk.show()

### 3. Il Muro del Pianto: I Titani più abbandonati 🧱
Essere famosi e longevi non salva dall'abbandono. Alcuni anime iconici hanno un "muro d'ingresso" altissimo.
Può essere dovuto a filler infiniti, cali di qualità o semplicemente una trama che si è persa.

In questo grafico isoliamo solo i **Titani (>10 anni)** e mostriamo la **Top 10 per Tasso di Abbandono**.
Questi sono i giganti che, nonostante la loro fama, hanno lasciato più "caduti" sul campo di battaglia.

In [264]:
# 1. Filtriamo solo i Titani (>10 anni)
titans_drop = df_drop[df_drop['duration_years'] >= 10].copy()

# 2. Prendiamo i 10 con il Drop Rate PEGGIORE (più alto)
# Filtriamo anche per popolarità minima (es. > 10.000 utenti) per evitare oscure serie sconosciute
titans_drop_major = titans_drop[titans_drop['total'] > 10000].sort_values(by='drop_rate', ascending=False).head(10)

# 3. Bar Chart
fig_wall = px.bar(titans_drop_major, 
                  y='title', 
                  x='drop_rate', 
                  color='drop_rate', # Più è rosso, più è alto il drop
                  orientation='h',
                  title='Gradico 13: Top 10 Titani per Tasso di Abbandono più alto',
                  labels={'title': '', 'drop_rate': 'Drop Rate (%)'},
                  color_continuous_scale='Reds',
                  text='duration_years', # Mostriamo da quanti anni vanno in onda
                  template='plotly_dark')

fig_wall.update_traces(texttemplate='%{x:.1f}% (da %{text:.0f} anni)', textposition='inside')
fig_wall.update_layout(yaxis={'categoryorder':'total ascending'}, xaxis_title="Percentuale Utenti che hanno abbandonato")
fig_wall.show()

# Capitolo 5: I Titani del Presente

In [265]:
# 1. DEFINIAMO df_ongoing
# Filtriamo solo le serie attualmente in corso
df_ongoing = df[df['status'] == 'Currently Airing'].copy()

# CORREZIONE: Usiamo tz='UTC' per renderlo compatibile con le date del dataset
current_date = pd.Timestamp.now(tz='UTC')
df_ongoing['current_duration_years'] = (current_date - df_ongoing['start_date']).dt.days / 365.25

# Pulizia dati
df_ongoing = df_ongoing[df_ongoing['current_duration_years'] > 0]

# Creiamo le categorie
df_ongoing['category'] = list(map(lambda x: 'Veterani (>10 anni)' if x >= 10 else 'Nuove Leve (<10 anni)', df_ongoing['current_duration_years']))


### 1. L'Abisso Temporale: I Pochissimi Eletti
Osservando il panorama degli anime attualmente in onda, notiamo una frattura netta. La quasi totalità delle produzioni odierne ha meno di 5 anni di vita.
Esiste però un ristretto club di **"Immortali"**: serie iniziate negli anni '60, '70 o '90 che continuano imperterrite.
Il grafico sottostante mostra questo "abisso": una nuvola densissima di nuove uscite (in basso a destra) e pochissimi punti isolati che si estendono verso l'alto a sinistra (Sazae-san, Doraemon, One Piece). Non c'è una distribuzione uniforme: o sei una hit del momento, o sei un'istituzione decennale.

In [266]:
# GRAFICO ALTERNATIVO MIGLIORATO (Fix NaN)

# Creiamo una copia temporanea dove rimuoviamo chi non ha un voto (score NaN)
# Questo risolve l'errore "Invalid elements include: [nan...]"
df_plot = df_ongoing.dropna(subset=['score']).copy()

fig_ongoing_better = px.scatter(df_plot, 
                                x='current_duration_years', 
                                y='members',
                                color='category', 
                                size='score', # Ora 'score' non ha più NaN
                                hover_name='title',
                                title='Grafico 14: I Veterani sono tutti Famosi?',
                                labels={'current_duration_years': 'Anni di Trasmissione', 'members': 'Popolarità (Members)', 'score': 'Voto'},
                                color_discrete_map={'Veterani (>10 anni)': '#EF553B', 'Nuove Leve (<10 anni)': '#636efa'},
                                log_y=True, # Scala logaritmica
                                template='plotly_dark')

fig_ongoing_better.update_layout(height=600)
fig_ongoing_better.show()

### 2. I Veri Re dello Schermo (Non sono chi credi)
Quando pensiamo agli anime infiniti, pensiamo a *One Piece* o *Naruto*. Ma la classifica reale per anni di trasmissione ci mostra una realtà diversa.
I veri dominatori non sono gli Shonen d'azione, ma gli anime per **Famiglie e Bambini** (Kids/Comedy/Slice of Life) come *Sazae-san*, *Nintama Rantarou* o *Doraemon*.
Queste serie sono pilastri culturali in Giappone, trasmessi in prima serata da decenni, con una funzione sociale rassicurante e abitudinaria che gli anime "da otaku" (serali/notturni) non possono eguagliare.

In [267]:
# Prendiamo i Top 15 attivi per durata
top_15_ongoing = df_ongoing.sort_values(by='current_duration_years', ascending=False).head(15)

fig_top_ongoing = px.bar(top_15_ongoing, 
                         y='title', 
                         x='current_duration_years', 
                         color='score', 
                         orientation='h',
                         title='<b>I Titani Viventi</b>: Le 15 Serie in Corso più Longeve',
                         labels={'title': 'Anime', 'current_duration_years': 'Anni in onda', 'score': 'Voto'},
                         color_continuous_scale='RdYlGn', # Usiamo la scala semantica richiesta
                         text='current_duration_years',
                         template='plotly_dark')

fig_top_ongoing.update_traces(texttemplate='%{text:.1f} anni', textposition='outside')
fig_top_ongoing.update_layout(yaxis={'categoryorder':'total ascending'}, height=700)
fig_top_ongoing.show()

## 🏁 Conclusioni: Il Segreto dell'Immortalità

In [268]:
# --- GRAFICO FINALE: "LA GALASSIA DELL'ANIMAZIONE" ---
# Uniamo tutte le dimensioni analizzate in un'unica visualizzazione globale.
# X: Storia (Start Year)
# Y: Qualità (Score)
# Dimensione: Longevità (Duration Years) -> Più è grande, più ha vissuto
# Colore: Stato (Attivo vs Finito)

# 1. Preparazione Dataset Finale
df_final = df.copy()

# Recalculate duration for everyone globaly just to be safe
calc_end_date = pd.to_datetime(df_final['end_date'].fillna(pd.Timestamp.now(tz='UTC')), utc=True)
start_date_final = pd.to_datetime(df_final['start_date'], errors='coerce', utc=True)
df_final['final_duration'] = (calc_end_date - start_date_final).dt.days / 365.25

# Clean Up
df_final = df_final.dropna(subset=['score', 'start_year', 'final_duration'])
df_final = df_final[df_final['final_duration'] > 0]
df_final = df_final[df_final['start_year'] >= 1960] # Focus sull'era moderna

# 2. Creazione della Galassia
fig_galaxy = px.scatter(df_final, 
                        x='start_year', 
                        y='score', 
                        size='final_duration', # La grandezza della stella è la sua vita
                        color='status',
                        hover_name='title',
                        hover_data={'final_duration': ':.1f', 'members': True},
                        title='<b>La Galassia dell\'Animazione</b>: Storia, Qualità e Sopravvivenza',
                        labels={'start_year': 'Anno di Nascita', 'score': 'Qualità (Score)', 'final_duration': 'Longevità (Anni)'},
                        color_discrete_map={'Currently Airing': '#00CC96', 'Finished Airing': '#EF553B', 'Not yet aired': 'gray'},
                        size_max=40, # Massimizziamo la differenza visiva
                        opacity=0.6, # Trasparenza per vedere le sovrapposizioni
                        template='plotly_dark')

fig_galaxy.update_layout(
    height=800,
    xaxis_title="Linea Temporale",
    yaxis_title="Rating (Qualità)",
    legend_title_text="Stato Vitale"
)

fig_galaxy.show()

Siamo partiti con una domanda provocatoria: *gli anime longevi sono opere d'arte che resistono al tempo o prodotti commerciali che sopravvivono per inerzia?*
I dati hanno parlato chiaro, smentendo alcuni pregiudizi e confermando nuove verità:

### 1. Il Darwinismo è Reale (Premessa Smentita)
L'idea che le serie lunghe "trascinino" stancamente i voti per abitudine è **falsa**. I dati mostrano una correlazione positiva tra longevità e punteggio.
Il mercato dell'anime è spietato: non c'è spazio per mediocrità duratura. Le opere che superano il decennio (i "Titani") sono quasi esclusivamente eccellenze (Score > 7.5). Se la qualità cala, la serie viene cancellata molto prima di diventare "immortale". La longevità non è un caso, è un **certificato di qualità**.

### 2. I Veri Immortali sono Invisibili
Abbiamo scoperto che i veri dominatori del tempo non sono le hit globali come *One Piece* o *Naruto*, ma le serie per famiglie (*Sazae-san*, *Doraemon*).
Questi anime operano su un livello diverso: non puntano all'hype o al trend del momento, ma all'indispensabilità sociale. Sono diventati riti quotidiani, immuni alle mode.

### 3. L'Esplosione Effimera
L'analisi storica mostra un'esplosione produttiva negli ultimi dieci anni (la "Bolla dello Streaming"), ma accompagnata da una frammentazione della durata. Si produce di più, ma si consuma (e si dimentica) più in fretta.
La "Galassia" qui sotto lo dimostra visivamente: una densità enorme di piccole stelle recenti di alta qualità ma vita breve, contro poche, giganti stelle antiche che continuano a bruciare.